In [13]:
%%html
<link href="https://fonts.googleapis.com/css2?family=Source+Serif+4:wght@300;400;500;600;700&display=swap" rel="stylesheet">

<style>
/* markdown + output */
body, .markdown-body, .jp-RenderedHTMLCommon, div.text_cell_render,
.notebook, .notebook * {
  font-family: "Source Serif 4", serif !important;
}

/* code editor (monaco) */
.monaco-editor, .monaco-editor * {
  font-family: "Source Serif 4", serif !important;
}
</style>

### paired t-test

paired t-test is used to compare the average of paired differences between two related samples (typically before and after measurements on the same subjects).

use cases:
- before and after treatment
- same subjects measured twice
- matched pairs experiments

formula: t = (d̄ - d₀) / (s_d / √n)

where:
- d̄ = mean of differences
- d₀ = claimed difference (often 0)
- s_d = standard deviation of differences
- n = number of pairs

In [14]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.stats import norm, t, binom, poisson, expon, chi2, f
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels
import statsmodels.api as sm
from statsmodels import stats as sm_stats
from statsmodels.stats import weightstats as sswa
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats import proportion as ssp
from statsmodels.stats.proportion import proportions_ztest
import os
import warnings
warnings.filterwarnings("ignore")

In [15]:
os.chdir(r'/Users/proxim/Desktop/CDAC-DBDA-coursework/08.advanced-analytics-stats')

### example: process improvement study

claim: after process revamp, turnaround time (tat) has come down by average of at least 0.7 minutes

hypotheses:
- H₀: μ_before - μ_after ≥ 0.7 (claim)
- H₁: μ_before - μ_after < 0.7 (left tail test)

In [16]:
# data: same subjects measured before and after
before = np.array([5, 8, 7, 6, 9, 8, 7])
after = np.array([4, 9, 5, 5, 9, 8, 6])

# calculate differences
diff = before - after
print('paired data:')
print(f'before: {before}')
print(f'after:  {after}')
print(f'diff:   {diff}')
print(f'\nmean difference = {np.mean(diff):.2f} minutes')
print(f'std of differences = {np.std(diff, ddof=1):.2f}')

paired data:
before: [5 8 7 6 9 8 7]
after:  [4 9 5 5 9 8 6]
diff:   [ 1 -1  2  1  0  0  1]

mean difference = 0.57 minutes
std of differences = 0.98


In [17]:
# method 1: using ttost_paired (two one-sided test)
result = sswa.ttost_paired(before, after, low=0.7, upp=0.7)


print(result)
# extract numeric p-values from the returned tuples
t_stat = result[0]
pval_left = result[1][1]
pval_right = result[2][1]

print('paired t-test results (ttost_paired):')
print(f'test statistic = {t_stat:.4f}')
print(f'p-value (left tail) = {pval_left:.4f}')
print(f'p-value (right tail) = {pval_right:.4f}')
print(f'\nfor left tail test, use left p-value: {pval_left:.4f}')
if pval_left < 0.05:
    print('reject H₀')
else:
    print('fail to reject H₀')

(np.float64(0.6303407448408251), (np.float64(-0.34856850115866744), np.float64(0.6303407448408251), np.float64(6.0)), (np.float64(-0.34856850115866744), np.float64(0.3696592551591749), np.float64(6.0)))
paired t-test results (ttost_paired):
test statistic = 0.6303
p-value (left tail) = 0.6303
p-value (right tail) = 0.3697

for left tail test, use left p-value: 0.6303
fail to reject H₀


In [18]:
# method 2: using scipy ttest_1samp on differences
t_stat, p_val = scipy.stats.ttest_1samp(before - after, popmean=0.7, alternative='less')
print('paired t-test results (ttest_1samp):')
print(f't-statistic = {t_stat:.4f}')
print(f'p-value = {p_val:.4f}')
print(f'\ninterpretation:')
if p_val < 0.05:
    print('reject H₀: evidence that improvement is less than 0.7 min')
else:
    print('fail to reject H₀: claim is reasonable')

paired t-test results (ttest_1samp):
t-statistic = -0.3486
p-value = 0.3697

interpretation:
fail to reject H₀: claim is reasonable


### one-way anova (analysis of variance)

one-way anova is used when comparing means of three or more independent groups to determine if at least one group mean is different.

hypotheses:
- H₀: all group means are equal (μ₁ = μ₂ = μ₃ = ...)
- H₁: at least one group mean is different

key concepts:
- total variance = between-group variance + within-group variance
- f-statistic = (between-group variance) / (within-group variance)
- larger f-statistic indicates more evidence against H₀
- p-value tells us probability of observing this f-statistic if H₀ is true

In [19]:
# load marks data
df = pd.read_excel('data/Marks.xlsx')
df.head()

,Subject,Trainer,Marks
0,Calculus,Sudeep,65
1,Calculus,Sudeep,75
2,Calculus,Sudeep,86
3,Calculus,Sudeep,82
4,Calculus,Sudeep,75


In [20]:
# check unique subjects
print('subjects in dataset:')
print(np.unique(df.Subject))
print(f'\ntrainers in dataset:')
print(np.unique(df.Trainer))

subjects in dataset:
['Calculus' 'Statistics' 'Trigno']

trainers in dataset:
['Ruchi' 'Sudeep']


### example: comparing marks across subjects

question: do students score differently in different subjects?

hypotheses:
- H₀: mean marks are equal across all subjects
- H₁: at least one subject has different mean marks

In [21]:
# perform one-way anova for marks by subject
# formula: 'dependent_variable ~ independent_variable'
model = ols('Marks ~ Subject', df).fit()
anova_table = sm.stats.anova_lm(model)
print('one-way anova: marks by subject')
print(anova_table)
print(f'\np-value = {anova_table["PR(>F)"][0]:.4f}')
print(f'\ninterpretation:')
if anova_table['PR(>F)'][0] < 0.05:
    print('reject H₀: at least one subject has different mean marks')
else:
    print('fail to reject H₀: all subjects have equal mean marks')

one-way anova: marks by subject
            df       sum_sq    mean_sq         F    PR(>F)
Subject    2.0    15.158906   7.579453  0.082763  0.920735
Residual  38.0  3480.060606  91.580542       NaN       NaN

p-value = 0.9207

interpretation:
fail to reject H₀: all subjects have equal mean marks


In [22]:
# perform one-way anova for marks by trainer
model_trainer = ols('Marks ~ Trainer', df).fit()
anova_trainer = sm.stats.anova_lm(model_trainer)
print('one-way anova: marks by trainer')
print(anova_trainer)
print(f'\np-value = {anova_trainer["PR(>F)"][0]:.4f}')
print(f'\ninterpretation:')
if anova_trainer['PR(>F)'][0] < 0.05:
    print('reject H₀: trainers have different mean student marks')
else:
    print('fail to reject H₀: trainers have equal mean student marks')

one-way anova: marks by trainer
            df       sum_sq      mean_sq         F    PR(>F)
Trainer    1.0  1007.318546  1007.318546  15.79059  0.000296
Residual  39.0  2487.900966    63.792332       NaN       NaN

p-value = 0.0003

interpretation:
reject H₀: trainers have different mean student marks


### anova background calculation - manual f-statistic

understanding how f-statistic is calculated helps interpret anova results better.

steps:
1. calculate grand mean (overall mean)
2. calculate total variance
3. calculate within-group variance
4. calculate between-group variance
5. f-statistic = (between variance / df_between) / (within variance / df_within)

In [23]:
# step 1: calculate grand mean
grand_mean = np.mean(df.Marks)
print(f'grand mean = {grand_mean:.2f}')

grand mean = 74.34


In [24]:
# step 2: calculate total variance
df['total_var'] = (df['Marks'] - grand_mean)**2
total_variance = np.sum(df.total_var)
print(f'total variance (ss_total) = {total_variance:.2f}')

total variance (ss_total) = 3495.22


In [25]:
# step 3: separate data by subject
df_calc = df[df.Subject == 'Calculus'].copy()
df_stats = df[df.Subject == 'Statistics'].copy()
df_trigno = df[df.Subject == 'Trigno'].copy()

# calculate group means
mean_calc = np.mean(df_calc.Marks)
mean_stats = np.mean(df_stats.Marks)
mean_trigno = np.mean(df_trigno.Marks)

print(f'calculus mean = {mean_calc:.2f}')
print(f'statistics mean = {mean_stats:.2f}')
print(f'trigonometry mean = {mean_trigno:.2f}')

calculus mean = 73.55
statistics mean = 75.07
trigonometry mean = 74.20


In [26]:
# step 4: calculate within-group variance
df_calc['within_var'] = (df_calc['Marks'] - mean_calc)**2
df_stats['within_var'] = (df_stats['Marks'] - mean_stats)**2
df_trigno['within_var'] = (df_trigno['Marks'] - mean_trigno)**2

ss_within = sum(df_calc['within_var']) + sum(df_stats['within_var']) + sum(df_trigno['within_var'])
print(f'within-group variance (ss_within) = {ss_within:.2f}')

within-group variance (ss_within) = 3480.06


In [27]:
# step 5: calculate between-group variance
n_calc = len(df_calc)
n_stats = len(df_stats)
n_trigno = len(df_trigno)

ss_between = (n_calc * (mean_calc - grand_mean)**2 + 
              n_stats * (mean_stats - grand_mean)**2 + 
              n_trigno * (mean_trigno - grand_mean)**2)

print(f'between-group variance (ss_between) = {ss_between:.2f}')
print(f'\nverification: ss_total = ss_between + ss_within')
print(f'{total_variance:.2f} = {ss_between:.2f} + {ss_within:.2f}')

between-group variance (ss_between) = 15.16

verification: ss_total = ss_between + ss_within
3495.22 = 15.16 + 3480.06


In [28]:
# step 6: calculate f-statistic
k = 3  # number of groups
n = len(df)  # total observations
df_between = k - 1
df_within = n - k

ms_between = ss_between / df_between
ms_within = ss_within / df_within
f_statistic = ms_between / ms_within

print(f'degrees of freedom (between) = {df_between}')
print(f'degrees of freedom (within) = {df_within}')
print(f'mean square (between) = {ms_between:.2f}')
print(f'mean square (within) = {ms_within:.2f}')
print(f'f-statistic = {f_statistic:.4f}')

degrees of freedom (between) = 2
degrees of freedom (within) = 38
mean square (between) = 7.58
mean square (within) = 91.58
f-statistic = 0.0828


In [29]:
# step 7: calculate p-value
p_value = 1 - f.cdf(f_statistic, df_between, df_within)
print(f'p-value = {p_value:.4f}')
print(f'\ninterpretation:')
if p_value < 0.05:
    print('reject H₀: group means are different')
else:
    print('fail to reject H₀: group means are equal')

p-value = 0.9207

interpretation:
fail to reject H₀: group means are equal


### post-hoc test: tukey hsd

after finding significant anova result, tukey hsd (honestly significant difference) test identifies which specific pairs of groups differ.

it performs pairwise comparisons while controlling for multiple testing error.

In [30]:
# tukey hsd for subject comparison
print('tukey hsd: pairwise subject comparison')
print(pairwise_tukeyhsd(df.Marks, df.Subject))
print(f'\ninterpretation:')
print('reject column shows if that pair has significantly different means')
print('false = means are not significantly different')
print('true = means are significantly different')

tukey hsd: pairwise subject comparison
    Multiple Comparison of Means - Tukey HSD, FWER=0.05     
  group1     group2   meandiff p-adj   lower   upper  reject
------------------------------------------------------------
  Calculus Statistics   1.5212 0.9156 -7.7434 10.7858  False
  Calculus     Trigno   0.6545 0.9838 -8.6101  9.9192  False
Statistics     Trigno  -0.8667 0.9667 -9.3889  7.6555  False
------------------------------------------------------------

interpretation:
reject column shows if that pair has significantly different means
false = means are not significantly different
true = means are significantly different
    Multiple Comparison of Means - Tukey HSD, FWER=0.05     
  group1     group2   meandiff p-adj   lower   upper  reject
------------------------------------------------------------
  Calculus Statistics   1.5212 0.9156 -7.7434 10.7858  False
  Calculus     Trigno   0.6545 0.9838 -8.6101  9.9192  False
Statistics     Trigno  -0.8667 0.9667 -9.3889  7.6555  Fal

In [31]:
# tukey hsd for trainer comparison
print('tukey hsd: pairwise trainer comparison')
print(pairwise_tukeyhsd(df.Marks, df.Trainer))
print(f'\ninterpretation:')
print('this shows whether the two trainers have significantly different student marks')

tukey hsd: pairwise trainer comparison
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj  lower   upper  reject
---------------------------------------------------
 Ruchi Sudeep   9.9879 0.0003 4.9039 15.0719   True
---------------------------------------------------

interpretation:
this shows whether the two trainers have significantly different student marks


### two-way anova

two-way anova analyzes the effect of two independent categorical variables on a continuous dependent variable.

advantages:
- tests main effect of each factor
- tests interaction effect between factors
- more efficient than running separate one-way anovas

example: marks depend on both subject and trainer

In [32]:
# two-way anova without interaction
model_2way = ols('Marks ~ Trainer + Subject', df).fit()
anova_2way = sm.stats.anova_lm(model_2way)
print('two-way anova (no interaction):')
print(anova_2way)
print(f'\ninterpretation:')
print(f'trainer effect p-value = {anova_2way["PR(>F)"][0]:.4f}')
print(f'subject effect p-value = {anova_2way["PR(>F)"][1]:.4f}')

two-way anova (no interaction):
            df       sum_sq      mean_sq          F    PR(>F)
Trainer    1.0  1007.318546  1007.318546  15.341424  0.000372
Subject    2.0    58.479354    29.239677   0.445319  0.644008
Residual  37.0  2429.421612    65.660044        NaN       NaN

interpretation:
trainer effect p-value = 0.0004
subject effect p-value = 0.6440


### interaction effect in two-way anova

interaction effect occurs when the effect of one factor depends on the level of another factor.

example: perhaps certain trainers are better at teaching certain subjects.

hypothesis:
- H₀: there is no interaction between trainer and subject
- H₁: there is interaction between trainer and subject

In [33]:
# two-way anova with interaction (use * instead of +)
model_interaction = ols('Marks ~ Trainer * Subject', df).fit()
anova_interaction = sm.stats.anova_lm(model_interaction)
print('two-way anova (with interaction):')
print(anova_interaction)
print(f'\ninterpretation:')
print(f'trainer effect p-value = {anova_interaction["PR(>F)"][0]:.4f}')
print(f'subject effect p-value = {anova_interaction["PR(>F)"][1]:.4f}')
print(f'interaction effect p-value = {anova_interaction["PR(>F)"][2]:.4f}')

two-way anova (with interaction):
                   df       sum_sq      mean_sq          F    PR(>F)
Trainer           1.0  1007.318546  1007.318546  15.892045  0.000325
Subject           2.0    58.479354    29.239677   0.461302  0.634241
Trainer:Subject   2.0   210.943834   105.471917   1.663987  0.204016
Residual         35.0  2218.477778    63.385079        NaN       NaN

interpretation:
trainer effect p-value = 0.0003
subject effect p-value = 0.6342
interaction effect p-value = 0.2040


### one-sample variance test (chi-square test)

this test compares sample variance against a claimed value to determine if the difference is statistically significant.

test statistic: χ² = (n-1) * s² / σ₀²

where:
- n = sample size
- s² = sample variance
- σ₀² = claimed population variance

the test statistic follows chi-square distribution with df = n-1

In [34]:
# load eruption data
df_faithful = pd.read_excel('data/CDAC_DataBook.xlsx', sheet_name='faithful')
erupt = df_faithful.eruptions

print(f'sample size = {len(erupt)}')
print(f'sample variance = {np.var(erupt, ddof=1):.4f}')

sample size = 272
sample variance = 1.3027


### example: left tail variance test

claim: minimum variance is 1.42

hypotheses:
- H₀: σ² ≥ 1.42
- H₁: σ² < 1.42 (left tail test)

In [35]:
# case 1: left tail test
claim_var = 1.42
test_stat = (len(erupt) - 1) * np.var(erupt, ddof=1) / claim_var
p_value = chi2.cdf(test_stat, len(erupt) - 1)

print(f'left tail variance test:')
print(f'claimed variance = {claim_var}')
print(f'sample variance = {np.var(erupt, ddof=1):.4f}')
print(f'chi-square statistic = {test_stat:.4f}')
print(f'p-value = {p_value:.4f}')
print(f'\ninterpretation:')
if p_value < 0.05:
    print('reject H₀: evidence that variance < 1.42')
else:
    print('fail to reject H₀: difference can be attributed to chance variation')

left tail variance test:
claimed variance = 1.42
sample variance = 1.3027
chi-square statistic = 248.6193
p-value = 0.1685

interpretation:
fail to reject H₀: difference can be attributed to chance variation


### example: right tail variance test

claim: maximum variance is 1.1

hypotheses:
- H₀: σ² ≤ 1.1
- H₁: σ² > 1.1 (right tail test)

In [36]:
# case 2: right tail test
claim_var2 = 1.1
test_stat2 = (len(erupt) - 1) * np.var(erupt, ddof=1) / claim_var2
p_value2 = 1 - chi2.cdf(test_stat2, len(erupt) - 1)

print(f'right tail variance test:')
print(f'claimed variance = {claim_var2}')
print(f'sample variance = {np.var(erupt, ddof=1):.4f}')
print(f'chi-square statistic = {test_stat2:.4f}')
print(f'p-value = {p_value2:.4f}')
print(f'\ninterpretation:')
if p_value2 < 0.05:
    print('reject H₀: evidence that variance > 1.1')
else:
    print('fail to reject H₀: claim is reasonable')

right tail variance test:
claimed variance = 1.1
sample variance = 1.3027
chi-square statistic = 320.9449
p-value = 0.0200

interpretation:
reject H₀: evidence that variance > 1.1


### example: two-tail variance test

claim: variance equals 1.5

hypotheses:
- H₀: σ² = 1.5
- H₁: σ² ≠ 1.5 (two tail test)

In [37]:
# case 3: two tail test
claim_var3 = 1.5
test_stat3 = (len(erupt) - 1) * np.var(erupt, ddof=1) / claim_var3
p1 = chi2.cdf(test_stat3, len(erupt) - 1)
p2 = 1 - chi2.cdf(test_stat3, len(erupt) - 1)

# for two-tail test, take smaller p-value and multiply by 2
p_value3 = min(p1, p2) * 2

print(f'two tail variance test:')
print(f'claimed variance = {claim_var3}')
print(f'sample variance = {np.var(erupt, ddof=1):.4f}')
print(f'chi-square statistic = {test_stat3:.4f}')
print(f'p-value (left) = {p1:.4f}')
print(f'p-value (right) = {p2:.4f}')
print(f'final p-value = {p_value3:.4f}')
print(f'\ninterpretation:')
if p_value3 < 0.05:
    print('reject H₀: evidence that variance ≠ 1.5')
else:
    print('fail to reject H₀: variance could be 1.5')

two tail variance test:
claimed variance = 1.5
sample variance = 1.3027
chi-square statistic = 235.3596
p-value (left) = 0.0577
p-value (right) = 0.9423
final p-value = 0.1153

interpretation:
fail to reject H₀: variance could be 1.5


### two-sample variance test (f-test)

this test compares variances of two independent samples to determine if their ratio is significantly different from a claimed value.

test statistic: f = (s₁² / s₂²) / claimed_ratio

the test statistic follows f-distribution with df₁ = n₁-1 and df₂ = n₂-1

In [38]:
# create two samples from eruption data
sample1 = erupt[30:75]
sample2 = erupt[130:185]

print(f'sample 1:')
print(f'  size = {len(sample1)}')
print(f'  variance = {np.var(sample1, ddof=1):.4f}')
print(f'\nsample 2:')
print(f'  size = {len(sample2)}')
print(f'  variance = {np.var(sample2, ddof=1):.4f}')
print(f'\nvariance ratio = {np.var(sample1, ddof=1) / np.var(sample2, ddof=1):.4f}')

sample 1:
  size = 45
  variance = 1.5416

sample 2:
  size = 55
  variance = 1.3054

variance ratio = 1.1809


### example: two-sample variance comparison

claim: variance of sample1 is at least 1.4 times variance of sample2

hypotheses:
- H₀: σ₁² / σ₂² ≥ 1.4
- H₁: σ₁² / σ₂² < 1.4 (left tail test)

In [39]:
# perform f-test
claimed_ratio = 1.4
f_stat = (np.var(sample1, ddof=1) / np.var(sample2, ddof=1)) / claimed_ratio
p_value_f = f.cdf(f_stat, len(sample1) - 1, len(sample2) - 1)

print(f'two-sample variance test:')
print(f'claimed ratio = {claimed_ratio}')
print(f'observed ratio = {np.var(sample1, ddof=1) / np.var(sample2, ddof=1):.4f}')
print(f'f-statistic = {f_stat:.4f}')
print(f'p-value = {p_value_f:.4f}')
print(f'\ninterpretation:')
if p_value_f < 0.05:
    print('reject H₀: evidence that ratio < 1.4')
else:
    print('fail to reject H₀: claim is reasonable')

two-sample variance test:
claimed ratio = 1.4
observed ratio = 1.1809
f-statistic = 0.8435
p-value = 0.2817

interpretation:
fail to reject H₀: claim is reasonable


### tests for discrete data

when dealing with categorical or count data, we use different tests:
- proportion tests (z-test for proportions)
- chi-square tests (goodness of fit, independence)
- poisson rate tests

note: discrete data can be approximated as continuous for large samples (n > 10)

### one-sample proportion test

this test compares sample proportion against a claimed value to determine if the difference is statistically significant.

commonly used for:
- success/failure rates
- approval ratings
- conversion rates
- yes/no survey responses

In [40]:
# example: percentage of freshers
# claim: percentage of freshers is limited to 70%
# H₀: p ≤ 0.7
# H₁: p > 0.7 (right tail test)

total_students = 75
freshers = 56
sample_prop = freshers / total_students

print(f'sample proportion = {sample_prop:.3f} or {sample_prop*100:.1f}%')
print(f'claimed proportion = 0.7 or 70%')

sample proportion = 0.747 or 74.7%
claimed proportion = 0.7 or 70%


In [41]:
# perform proportion z-test
z_stat_prop, p_val_prop = ssp.proportions_ztest(freshers, total_students, value=0.7, alternative='larger')

print('one-sample proportion test:')
print(f'z-statistic = {z_stat_prop:.4f}')
print(f'p-value = {p_val_prop:.4f}')
print(f'\ninterpretation:')
if p_val_prop < 0.05:
    print('reject H₀: evidence that proportion > 70%')
else:
    print('fail to reject H₀: proportion ≤ 70% is reasonable')

one-sample proportion test:
z-statistic = 0.9292
p-value = 0.1764

interpretation:
fail to reject H₀: proportion ≤ 70% is reasonable


### two-sample proportion test

this test compares proportions from two independent samples to determine if their difference is statistically significant.

examples:
- comparing conversion rates between two marketing campaigns
- comparing success rates of two treatments
- comparing approval ratings between two groups

### approximating discrete as continuous

for large sample sizes (n > 10), discrete distributions can be approximated by continuous distributions:
- binomial → normal
- poisson → normal

this simplifies calculations and provides good approximations.

In [42]:
# example: binomial approximation
# probability of 200 rainy days out of 300, when p(rain) = 0.7

# exact using binomial
prob_binom = binom.cdf(200, 300, 0.7)
print(f'exact probability (binomial) = {prob_binom:.4f}')

# approximation using normal
mean_days = 300 * 0.7
std_days = np.sqrt(300 * 0.7 * (1 - 0.7))
prob_norm = norm.cdf(200, mean_days, std_days)
print(f'approximate probability (normal) = {prob_norm:.4f}')

# with continuity correction (200.5 instead of 200)
prob_norm_corrected = norm.cdf(200.5, mean_days, std_days)
print(f'approximate with correction = {prob_norm_corrected:.4f}')
print(f'\ncontinuity correction improves accuracy')

exact probability (binomial) = 0.1163
approximate probability (normal) = 0.1039
approximate with correction = 0.1157

continuity correction improves accuracy


In [44]:
# probability of incident between 3 to 5 hours
# can only use exponential (poisson gives count, not time intervals)
prob_interval = expon.cdf(5, scale=0.5) - expon.cdf(3, scale=0.5)
print(f'probability of incident between 3-5 hours = {prob_interval:.4f}')

probability of incident between 3-5 hours = 0.0024
